In [2]:
print("hello")

hello


In [3]:
import pandas as pd
import numpy as np

df=pd.read_csv("machine_rul_dataset.csv")
print(df)

    Record_ID Machine_Type Operating_Mode Lubrication_Type  Temperature_C  \
0    REC_1000       Type_C            Eco          Mineral          85.80   
1    REC_1001       Type_A       Standard          Mineral          78.69   
2    REC_1002       Type_C            Eco          Mineral          84.75   
3    REC_1003       Type_C       Standard          Mineral          82.56   
4    REC_1004       Type_A       Standard        Synthetic          65.05   
..        ...          ...            ...              ...            ...   
195  REC_1195       Type_B       Standard        Synthetic          92.76   
196  REC_1196       Type_C       Standard          Mineral          75.93   
197  REC_1197       Type_C      High_Load          Mineral          64.66   
198  REC_1198       Type_B      High_Load        Synthetic          93.28   
199  REC_1199       Type_A      High_Load          Mineral          81.47   

     Vibration_mm_s  Pressure_kPa     RPM  Operating_Hours  \
0            

In [4]:
df.head()

,Record_ID,Machine_Type,Operating_Mode,Lubrication_Type,Temperature_C,Vibration_mm_s,Pressure_kPa,RPM,Operating_Hours,Remaining_Useful_Life_Hours
0,REC_1000,Type_C,Eco,Mineral,85.80,1.348,91.40,3081.2,2282.0,3137.38
1,REC_1001,Type_A,Standard,Mineral,78.69,0.705,91.80,2393.7,2622.6,3224.85
2,REC_1002,Type_C,Eco,Mineral,84.75,1.001,99.51,3619.0,1860.2,3496.60
3,REC_1003,Type_C,Standard,Mineral,82.56,1.177,91.85,3718.4,3005.5,2975.30
4,REC_1004,Type_A,Standard,Synthetic,65.05,2.057,89.31,2754.9,901.3,3931.38


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Record_ID                    200 non-null    str    
 1   Machine_Type                 195 non-null    str    
 2   Operating_Mode               200 non-null    str    
 3   Lubrication_Type             200 non-null    str    
 4   Temperature_C                195 non-null    float64
 5   Vibration_mm_s               200 non-null    float64
 6   Pressure_kPa                 195 non-null    float64
 7   RPM                          200 non-null    float64
 8   Operating_Hours              200 non-null    float64
 9   Remaining_Useful_Life_Hours  200 non-null    float64
dtypes: float64(6), str(4)
memory usage: 15.8 KB


In [6]:
df.columns

Index(['Record_ID', 'Machine_Type', 'Operating_Mode', 'Lubrication_Type',
       'Temperature_C', 'Vibration_mm_s', 'Pressure_kPa', 'RPM',
       'Operating_Hours', 'Remaining_Useful_Life_Hours'],
      dtype='str')

In [7]:
df.isnull().sum().sum()

np.int64(15)

In [8]:
df.isnull().sum()

Record_ID                      0
Machine_Type                   5
Operating_Mode                 0
Lubrication_Type               0
Temperature_C                  5
Vibration_mm_s                 0
Pressure_kPa                   5
RPM                            0
Operating_Hours                0
Remaining_Useful_Life_Hours    0
dtype: int64

In [28]:
if "Record_ID" in df.columns:
    df.drop(columns=["Record_ID"], inplace=True)
X=df.drop(columns=["Remaining_Useful_Life_Hours"])

In [29]:
y=df["Remaining_Useful_Life_Hours"]

In [30]:
print(X.shape)

(200, 8)


In [31]:
print(y.shape)

(200,)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

In [33]:
print(num_cols)

['Temperature_C', 'Vibration_mm_s', 'Pressure_kPa', 'RPM', 'Operating_Hours']


In [34]:
print(cat_cols)

['Machine_Type', 'Operating_Mode', 'Lubrication_Type']


In [35]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [36]:
print(df.isnull().sum())

Machine_Type                   0
Operating_Mode                 0
Lubrication_Type               0
Temperature_C                  0
Vibration_mm_s                 0
Pressure_kPa                   0
RPM                            0
Operating_Hours                0
Remaining_Useful_Life_Hours    0
dtype: int64


In [37]:
for nums in num_cols:
    df[nums]=df[nums].fillna(df[nums].median())

In [38]:
print(df.isnull().sum())

Machine_Type                   0
Operating_Mode                 0
Lubrication_Type               0
Temperature_C                  0
Vibration_mm_s                 0
Pressure_kPa                   0
RPM                            0
Operating_Hours                0
Remaining_Useful_Life_Hours    0
dtype: int64


In [39]:
for cats in cat_cols:
    df[cats]=df[cats].fillna(df[cats].mode()[0])

In [40]:
print(df.isnull().sum())

Machine_Type                   0
Operating_Mode                 0
Lubrication_Type               0
Temperature_C                  0
Vibration_mm_s                 0
Pressure_kPa                   0
RPM                            0
Operating_Hours                0
Remaining_Useful_Life_Hours    0
dtype: int64


In [41]:
print(X_train.shape)

(160, 8)


In [42]:
print(X_test.shape)

(40, 8)


In [43]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [44]:
X_train_encoded=preprocessor.fit_transform(X_train)
X_test_encoded=preprocessor.transform(X_test)

In [45]:
print(X_train_encoded.shape)

(160, 13)


In [46]:
print(X_test_encoded.shape)

(40, 13)


In [52]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC, SVR
svr=SVR(kernel='rbf')
param_grid={
    'C':[0.1,0.6,0.8,1],
    'gamma':['scale','auto'],
}

grid=GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=1
)

grid.fit(X_train_encoded,y_train)

best_model1=grid.best_estimator_
svr_df=grid.cv_results_
best_score1=grid.best_score_
best_params1=grid.best_params_

print(best_model1)
print(svr_df)
print(best_score1)
print(best_params1)





SVR(C=1, gamma='auto')
{'mean_fit_time': array([0.00218344, 0.00107708, 0.0009007 , 0.00171762, 0.00172071,
       0.00177193, 0.00165486, 0.00156102]), 'std_fit_time': array([3.54473765e-04, 1.21447588e-04, 3.19072833e-05, 4.30139981e-04,
       3.78337470e-04, 1.22024296e-04, 2.61963112e-04, 3.67383229e-04]), 'mean_score_time': array([0.0013135 , 0.00071073, 0.00066471, 0.00112052, 0.00125318,
       0.00112734, 0.00121398, 0.00113869]), 'std_score_time': array([3.33989197e-04, 9.92363314e-05, 1.72971324e-04, 2.72678711e-04,
       2.89300684e-05, 2.06088534e-04, 1.59704532e-04, 1.46386911e-04]), 'param_C': masked_array(data=[0.1, 0.1, 0.6, 0.6, 0.8, 0.8, 1.0, 1.0],
             mask=[False, False, False, False, False, False, False, False],
       fill_value=1e+20), 'param_gamma': masked_array(data=['scale', 'auto', 'scale', 'auto', 'scale', 'auto',
                   'scale', 'auto'],
             mask=[False, False, False, False, False, False, False, False],
       fill_value=np.st

In [53]:
actual_mse=-best_score1
print(actual_mse)

625075.1574617319


In [54]:

actual_rmse = np.sqrt(-best_score1)
print("Actual RMSE:", actual_rmse)

Actual RMSE: 790.6169473656203


In [56]:
from sklearn.model_selection import KFold, cross_val_score

scores1=-cross_val_score(best_model1,X_train_encoded,y_train,cv=5,scoring='neg_root_mean_squared_error')

In [57]:
print(scores1)

[831.47781178 810.6808175  754.13426482 833.50093744 716.5017417 ]


In [58]:
from sklearn.decomposition import PCA
pca=PCA()
pca.fit(X_train_encoded)

cum_var=np.cumsum(pca.explained_variance_ratio_)
n_components_95=int(np.argmax(cum_var>=0.95)+1)

print("Original Features Count        :", X_train_encoded.shape[1])
print("Components for >= 95% Variance :", n_components_95)
print("Cumulative Explained Variance  :\n", np.round(cum_var, 4))


Original Features Count        : 13
Components for >= 95% Variance : 9
Cumulative Explained Variance  :
 [0.1693 0.3323 0.4808 0.6228 0.7477 0.8204 0.8742 0.923  0.964  1.
 1.     1.     1.    ]


In [68]:
pca_full = PCA(n_components=n_components_95, random_state=42,svd_solver='arpack')
X_train_pca = pca_full.fit_transform(X_train_encoded)
X_test_pca = pca_full.transform(X_test_encoded)

print("X_train_pca Shape              :", X_train_pca.shape)
print("X_test_pca Shape               :", X_test_pca.shape)

X_train_pca Shape              : (160, 9)
X_test_pca Shape               : (40, 9)


In [69]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
svr_pca = SVR(kernel="rbf")

param_grid = {
    'C': [0.1, 0.6, 0.8, 1],
    'gamma': ['scale', 'auto']
}

grid_pca = GridSearchCV(
    estimator=svr_pca,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

grid_pca.fit(X_train_pca, y_train)

best_model2 = grid_pca.best_estimator_
best_params2 = grid_pca.best_params_
best_score2 = grid_pca.best_score_

print("\n--- Model 2 (SVR with PCA) Tuning Results ---")
print("Best Model     :", best_model2)
print("Best Parameters:", best_params2)
print("Best CV Score  :", best_score2)
print("Actual CV RMSE :", -best_score2)

y_pred_pca = best_model2.predict(X_test_pca)

print("\n--- Model 2 (SVR with PCA) Test Set Evaluation ---")
print("R2 Score :", r2_score(y_test, y_pred_pca))
print("MAE      :", mean_absolute_error(y_test, y_pred_pca))
print("MSE      :", mean_squared_error(y_test, y_pred_pca))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, y_pred_pca)))

# ==============================================================================
# 4. 5-Fold Cross-Validation Scores for Model 2
# ==============================================================================
scores2 = -cross_val_score(
    best_model2,
    X_train_pca,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("\nScores with PCA across 5 folds:", scores2)
print("Mean RMSE with PCA             :", scores2.mean())


--- Model 2 (SVR with PCA) Tuning Results ---
Best Model     : SVR(C=1, gamma='auto')
Best Parameters: {'C': 1, 'gamma': 'auto'}
Best CV Score  : -789.1710346360297
Actual CV RMSE : 789.1710346360297

--- Model 2 (SVR with PCA) Test Set Evaluation ---
R2 Score : -0.0021144600767770694
MAE      : 699.738606266321
MSE      : 648571.7990964511
RMSE     : 805.339555154502

Scores with PCA across 5 folds: [831.43481347 810.62577728 753.90384554 833.48936956 716.40136733]
Mean RMSE with PCA             : 789.1710346360297


In [70]:
table_d=pd.DataFrame({
    "Fold":[1,2,3,4,5],
    "RMSE without PCA":scores1,
    "RMSE with PCA":scores2,
    "Difference":scores2-scores1
})

print("5 fold cross-validate")
print(table_d)

5 fold cross-validate
   Fold  RMSE without PCA  RMSE with PCA  Difference
0     1        831.477812     831.434813   -0.042998
1     2        810.680818     810.625777   -0.055040
2     3        754.134265     753.903846   -0.230419
3     4        833.500937     833.489370   -0.011568
4     5        716.501742     716.401367   -0.100374


In [71]:
print("\nMean RMSE Without PCA :", np.mean(scores1))
print("Mean RMSE With PCA    :", np.mean(scores2))


Mean RMSE Without PCA : 789.2591146498916
Mean RMSE With PCA    : 789.1710346360297


In [72]:
from scipy.stats import ttest_rel, ttest_ind, wilcoxon

paired_stat, paired_p = ttest_rel(scores1, scores2)
if paired_p < 0.05:
    print("Paired t-test: Significant difference")
else:
    print("Paired t-test: No significant difference")



Paired t-test: No significant difference


In [74]:
model_scores=[scores1,scores2]

In [75]:
model_means = [np.mean(scores) for scores in model_scores]
best_index = np.argmax(model_means) # or np.argmin if lower is better
print(f"Best model: Model {best_index + 1} with mean score = { model_means[best_index]:.4f}")


Best model: Model 1 with mean score = 789.2591
